[![Open in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/EelcoHoogendoorn/numga/blob/main/examples/math/fractals/fractals.ipynb)

# Fractals of the even subalgebra

The even multivectors of the plane, a scalar plus a multiple of $xy$, multiply among themselves, and since $xy$ squares to minus one they multiply as the complex numbers. Iterating $z \mapsto z^2 + c$ on them from $z = c$, for each $c$ of a grid, draws the Mandelbrot set: the $c$ for which the iteration stays bounded. The even multivectors of space, a scalar plus a bivector, multiply as the quaternions, and the same iteration, with $c$ fixed and the start varying, draws a Julia set of four dimensions, of which a plane through the scalar axis shows a slice. Turning the plane turns the slice.

In [ ]:
# The repository root on the path, for numga and the examples; in Colab, fetch the repository first.
import sys
from pathlib import Path

if "google.colab" in sys.modules:
    root = Path("/content/numga")
    if not root.exists():
        import subprocess
        subprocess.run(["git", "clone", "--depth", "1", "https://github.com/EelcoHoogendoorn/numga.git", str(root)], check=True)
else:
    root = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "numga").is_dir() and (p / "examples").is_dir())
sys.path.insert(0, str(root))

In [ ]:
%matplotlib inline

import numpy as np
from IPython.display import Image, display

from numga import Algebra, NumpyContext
from numga.algebras import VGA3D
from examples.animation import save_animation, save_figure
from examples.math.fractals import render

plane = NumpyContext(Algebra("x+y+")).multivector
space = NumpyContext(VGA3D).multivector
# The view of the Mandelbrot set, its corners as a scalar part and a part along xy.
MANDELBROT_LOWER, MANDELBROT_UPPER = (-2.2, -1.1), (0.8, 1.1)
MANDELBROT_ROWS, MANDELBROT_COLUMNS = 440, 600
MANDELBROT_STEPS = 120
# The Julia set's constant, and the view of its slices.
JULIA = -0.2 + 0.6 * space.yz + 0.2 * space.zx
JULIA_LOWER, JULIA_UPPER = (-1.5, -1.5), (1.5, 1.5)
JULIA_SIZE = 360
JULIA_STEPS = 60
FRAMES = 60
FRAME_MS = 60

## 1. The complex numbers in the plane

The even multivectors of the plane are a scalar and a multiple of $xy$, and $xy$ squares to minus one:

In [ ]:
print((plane.xy * plane.xy).coefficient_table())

## 2. The Mandelbrot set

Each point is iterated until it leaves the circle of radius two, and there it stops; the number of steps it stayed inside colours it. The iteration is the product of even multivectors, $z$ with itself, plus the constant.

In [ ]:
def escape(start, constant, steps: int, mv):
    """The points iterated by `z * z + constant` until they leave the circle of radius two, where
    they stop, and how many steps each stayed inside."""
    points = start
    count = mv.scalar(np.zeros(start.shape + (1,)))                             # [...] Scalar
    for _ in range(steps):
        room = 4 - points.scalar_norm_squared()                                 # [...] Scalar
        # One while inside the circle, zero once outside.
        inside = (room.abs() + room) / (2 * room.abs())                         # [...] Scalar
        points = points + inside * (points * points + constant - points)
        count = count + inside
    return points, count


across, down = render.grid(MANDELBROT_LOWER, MANDELBROT_UPPER, MANDELBROT_ROWS, MANDELBROT_COLUMNS)
constants = across + down * plane.xy                                            # [rows, columns] Even
mandelbrot = escape(constants, constants, MANDELBROT_STEPS, plane)

In [ ]:
save_figure(render.show(render.escape_image(*mandelbrot, MANDELBROT_STEPS), MANDELBROT_LOWER, MANDELBROT_UPPER), "mandelbrot")

## 3. A quaternion Julia set, slice by slice

The constant is a scalar plus a bivector, and the start runs over a plane of even multivectors, spanned by the scalar and a unit bivector. While that bivector is the direction of the constant's bivector, the scalar and it multiply as the complex numbers, and the slice is a complex Julia set. Turning the bivector out of that direction, by the rotor that turns its normal towards $z$, shows the other faces of the four-dimensional set.

In [ ]:
def slices(frames: int):
    """Per frame, the plane of starts turned a little further from the constant's direction towards
    xy, its points iterated, and their escape counts."""
    direction = JULIA.select_grade(2).normalized()                              # [] Bivector
    away = (direction.dual() ^ space.z).normalized()                            # [] Bivector
    across, down = render.grid(JULIA_LOWER, JULIA_UPPER, JULIA_SIZE, JULIA_SIZE)
    for angle in np.linspace(0.0, np.pi / 2, frames):
        turned = (away * (-angle / 2)).exp() >> direction                       # [] Bivector
        yield escape(across + down * turned, JULIA, JULIA_STEPS, space)


display(Image(filename=save_animation(render.animate(slices(FRAMES), JULIA_STEPS), "julia_slices", FRAME_MS)))

In [ ]:
# checks
# xy squares to minus one; the origin stays bounded and a constant beyond the circle escapes at once;
# the first slice, through the constant's own bivector, is the complex Julia set of the constant with
# that bivector's size as its imaginary part.
np.testing.assert_allclose((plane.xy * plane.xy).to_array(), -1.0)
origin = escape(plane.scalar([0.0]), plane.scalar([0.0]), MANDELBROT_STEPS, plane)[1]
np.testing.assert_allclose(origin.to_array(), MANDELBROT_STEPS, atol=1e-9)
far = escape(plane.scalar([2.5]), plane.scalar([2.5]), MANDELBROT_STEPS, plane)[1]
np.testing.assert_allclose(far.to_array(), 0.0)
first = next(slices(1))[1].to_array()
size = np.sqrt(0.6**2 + 0.2**2)
across, down = render.grid(JULIA_LOWER, JULIA_UPPER, JULIA_SIZE, JULIA_SIZE)
complex_julia = escape(across + down * plane.xy, -0.2 + size * plane.xy, JULIA_STEPS, plane)[1].to_array()
np.testing.assert_allclose(first, complex_julia, atol=1e-9)